# Experiment Setup: C-Core Configuration and Training

## Overview

This section details the complete experimental setup for training Deep Reinforcement Learning agents on the C-core electromagnetic actuator topology optimization problem.

## C-Core Problem Definition

### Physical Configuration
- **Domain Size**: 40×30 grid cells
- **Resolution**: 1.0 mm per cell
- **Material**: Magnetic steel with μr = 1000
- **Coil Region**: Central excitation area
- **Air Gap**: Optimization target region

### Design Objectives
1. **Primary**: Maximize electromagnetic force
2. **Secondary**: Minimize material usage
3. **Constraints**: Symmetry, manufacturability
4. **Physics**: Magnetostatic field analysis

## Environment Configuration

### SeqTO-v2 Settings
```yaml
environment:
  width: 40
  height: 30
  max_steps: 1200
  state_channels: 4
  action_type: "placement"
  reward_scaling: 100.0
  
state_representation:
  use_cnn_encoder: true
  cnn_channels: [32, 64, 128]
  include_gradients: true
  include_history: true
  
reward_function:
  force_weight: 1.0
  material_weight: 0.1
  smoothness_weight: 0.05
  constraint_penalty: 10.0
```

### FEMM Integration
- **Simulation Type**: Magnetostatic (DC)
- **Mesh Resolution**: 1.0 mm elements
- **Convergence Criteria**: 1e-8 precision
- **Boundary Conditions**: Dirichlet (A = 0)
- **Materials**: Air, Steel_1008, Copper

## Training Pipeline

### A2C Agent Configuration
```yaml
agent:
  algorithm: "A2C"
  actor_lr: 0.0003
  critic_lr: 0.001
  hidden_sizes: [512, 256, 128]
  activation: "relu"
  dropout_rate: 0.1
  
training:
  total_episodes: 5000
  max_steps_per_episode: 1200
  gamma: 0.99
  gae_lambda: 0.95
  entropy_coef: 0.01
  value_loss_coef: 0.5
  max_grad_norm: 0.5
  
advanced_features:
  curriculum_learning: true
  difficulty_progression: [0.3, 0.5, 0.7, 1.0]
  entropy_annealing: true
  lr_scheduling: true
```

### Training Protocol

#### Initialization
1. Set random seeds for reproducibility
2. Initialize networks with Xavier/Glorot initialization
3. Set up optimizers with specified learning rates
4. Configure logging and checkpointing

#### Training Loop
```python
for episode in range(total_episodes):
    # Update curriculum difficulty
    if episode % curriculum_interval == 0:
        update_curriculum_difficulty()
    
    # Run episode
    states, actions, rewards = run_episode()
    
    # Calculate advantages and returns
    advantages, returns = calculate_advantages_returns(rewards)
    
    # Update networks
    actor_loss, critic_loss = update_networks(states, actions, advantages, returns)
    
    # Logging and checkpointing
    if episode % log_interval == 0:
        log_metrics(episode, actor_loss, critic_loss, rewards)
    
    if episode % save_interval == 0:
        save_checkpoint(episode)
```

## Hyperparameter Optimization

### Search Space
```python
hyperparameter_space = {
    'actor_lr': [1e-5, 1e-3],
    'critic_lr': [1e-4, 1e-2],
    'hidden_sizes': [(128, 64), (256, 128), (512, 256)],
    'entropy_coef': [0.001, 0.1],
    'gamma': [0.95, 0.999],
    'batch_size': [16, 32, 64]
}
```

### Optimization Method
- **Search Algorithm**: Bayesian optimization
- **Evaluation Metric**: Composite performance score
- **Trial Budget**: 20 hyperparameter configurations
- **Early Stopping**: 50 episodes without improvement

## Experimental Protocol

### Reproducibility Measures
- **Random Seeds**: Fixed seeds for each run
- **Environment Setup**: Identical initial conditions
- **Hardware**: Consistent computational resources
- **Software Versions**: Fixed library versions

### Statistical Validation
- **Multiple Runs**: 5 independent runs per configuration
- **Statistical Tests**: T-test, Wilcoxon rank-sum
- **Effect Size**: Cohen's d calculation
- **Confidence Intervals**: 95% confidence level

### Baseline Comparisons
1. **Traditional GA**: Genetic algorithm optimization
2. **Random Search**: Random topology exploration
3. **DQN**: Deep Q-Network baseline
4. **A2C**: Primary method (this work)

## Monitoring and Diagnostics

### Training Metrics
- **Episode Reward**: Cumulative reward per episode
- **Loss Values**: Actor and critic losses
- **Entropy**: Policy entropy (exploration measure)
- **Gradient Norms**: Training stability indicator

### Performance Metrics
- **Final Force**: Electromagnetic force (N)
- **Material Efficiency**: Force per unit material
- **Convergence Episode**: Episode to 90% of max performance
- **Training Stability**: Performance variance

### Health Monitoring
```python
def check_training_health(metrics):
    warnings = []
    recommendations = []
    
    # Check for gradient explosion
    if metrics['grad_norm'] > 10.0:
        warnings.append("High gradient norm detected")
        recommendations.append("Reduce learning rate or gradient clipping")
    
    # Check for loss divergence
    if metrics['actor_loss'] > early_loss * 2:
        warnings.append("Actor loss divergence")
        recommendations.append("Check network architecture and learning rate")
    
    # Check for performance plateau
    if metrics['reward_variance'] < 0.01:
        warnings.append("Performance plateau detected")
        recommendations.append("Increase exploration or adjust learning rate")
    
    return warnings, recommendations
```

## Computational Requirements

### Hardware Specifications
- **CPU**: 8+ cores for physics simulation
- **RAM**: 16GB+ for batch processing
- **GPU**: CUDA-compatible for network training
- **Storage**: 10GB+ for checkpoints and logs

### Software Dependencies
- **Python**: 3.8+
- **PyTorch**: 1.12+
- **NumPy/SciPy**: For numerical computations
- **Matplotlib/Seaborn**: For visualization
- **FEMM**: For physics simulation

## Results Analysis Framework

### Performance Evaluation
- **Convergence Analysis**: Learning curve comparison
- **Final Performance**: Statistical significance testing
- **Design Quality**: Topology feature analysis
- **Computational Efficiency**: Time and resource usage

### Visualization Tools
- **Learning Curves**: Training progress plots
- **Topology Evolution**: Design progression visualization
- **Performance Distributions**: Statistical analysis plots
- **Comparison Charts**: Method comparison graphics

## Expected Outcomes

Based on preliminary experiments:
- **Convergence**: 2000-3000 episodes to optimal performance
- **Performance**: 30%+ improvement over traditional methods
- **Stability**: Low variance across multiple runs
- **Efficiency**: Reasonable computational requirements

## Conclusion

This comprehensive experimental setup provides a robust framework for evaluating Deep Reinforcement Learning approaches to C-core topology optimization. The systematic protocol ensures reproducible results and meaningful performance comparisons across different optimization methods.